Objectif :
- Parser une requete SQL
- Récupérer les alias et critères de jointure

In [ ]:
import re
import pandas as pd
from pathlib import Path
import sqlfluff
from pprint import pprint

# Lib bigquery_sql_parser

In [ ]:
from bigquery_sql_parser.script import Script
from bigquery_sql_parser.query import Query

In [ ]:
script = Script.from_file("sql/dm_market_chantiers.sql")
block = script.blocks
cte_list = script.ctes

for cte in cte_list:
    print("===================")
    print("name :", cte.name)
    print("text :", cte.text)
    break

    #query = Query(cte.text)
    #for column in query.columns:
    #    print("col :", column.value, column.name)

In [ ]:
query_content = Path("notebooks/query_groupby.sql").read_text()

query = Query(query_content)
[
    (column.find_name(), column.find_value().strip())
    for column in query.columns
]

In [ ]:
for tokens in groups:
    print("======================")
    for token in tokens:
        print("token :", token.ttype)
        print("value :", token.value)
        print("norm  :", token.normalized)
        print()

# Lib sqlparse

In [ ]:
import sqlparse
from gen_table_dependencies import (
    split_tokens,
    groups_to_usable,
    rename_dependencies
)

In [ ]:
stments = sqlparse.split(Path("sql/dm_market_chantiers.sql").read_text())
len(stments)

In [ ]:
query = Path("notebooks/query_groupby.sql").read_text()
print(query)

p = sqlparse.parse(query)[0]
groups = list(split_tokens(p.tokens))

tables_used, join_conditions = groups_to_usable(groups)
pprint(tables_used)

list(rename_dependencies(tables_used, join_conditions))

# regex parsing

In [ ]:
def clean_query(query):
    # replace "  " -> " "
    query_len = 0
    while len(query) != query_len:
        query_len = len(query)
        query = query.replace("  ", " ")
        
    return query

def make_alias(var_name: str):
    return f"(?P<{var_name}>[a-z0-9_]+)"

def extract_from(query):
    dataset_ex = make_alias("dataset")
    table_ex = make_alias("table_name")
    alias_ex = make_alias("alias")

    regex_pattern = f"\s+\`?{dataset_ex}\.{table_ex}\`?\s+(?:as)?\s+{alias_ex}?\s+(?:left|right|inner|cross)"
    regex_pattern = f"\s+\`?{dataset_ex}\.{table_ex}\`?\s+(?:as)?"

    return re.findall(r"from" + regex_pattern, query, flags=re.IGNORECASE)



for file_name, query, expected in get_query():
    extracted_from = extract_from(query)
    if extracted_from == expected["from"]:
        print(file_name , "ok")
    else:
        print(file_name, "ko, expecting", expected["from"], "got", extracted_from)
    
    

In [ ]:
re.findall("((?:left|inner|cross) join)", "cross join", flags=re.IGNORECASE)

In [ ]:
def make_alias(var_name: str):
    return f"(?P<{var_name}>[a-z0-9_]+)"

dataset_ex = make_alias("dataset")
table_ex = make_alias("table_name")
alias_ex = make_alias("alias")

regex_pattern = f"\s+\`?{dataset_ex}\.{table_ex} (as {alias_ex})?"
re.findall(r"join" + regex_pattern, query, flags=re.IGNORECASE)